# Замер скорости раннера
Эталонные модели на 5 фолдах: по этим временам планируем бюджет финальной тетради (≤ 10 минут).

In [ ]:
import os, time, sys
import numpy as np, pandas as pd, lightgbm as lgb, xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
print("CPU:", os.cpu_count())
tr = pd.read_csv("../data/train.csv"); y = np.log(tr.Price); X = tr.drop(columns=["Id", "Price"])
cats = ["Balcony", "Walls", "Age", "Lift", "District", "Okrug"]
Xc = X.copy()
for c in cats: Xc[c] = Xc[c].astype("category")
Xs = X.copy(); Xs[cats] = Xs[cats].fillna("NA")
folds = list(KFold(5, shuffle=True, random_state=0).split(X))
def bench(name, make, data):
    t = time.time(); oof = np.zeros(len(y))
    for a, b in folds:
        m = make(); m.fit(data.iloc[a], y.iloc[a]); oof[b] = m.predict(data.iloc[b])
    print(f"{name:10s} {time.time()-t:6.1f} s   RMSE {np.sqrt(((oof-y)**2).mean()):.4f}")

In [ ]:
bench("LightGBM", lambda: lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.03, num_leaves=63, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1), Xc)
bench("XGBoost", lambda: xgb.XGBRegressor(n_estimators=1500, learning_rate=0.03, max_depth=8, subsample=0.8, colsample_bytree=0.8, enable_categorical=True, tree_method="hist"), Xc)
bench("CatBoost", lambda: CatBoostRegressor(iterations=2000, learning_rate=0.08, depth=8, cat_features=cats, verbose=0, random_seed=0), Xs)